In [10]:
import pandas as pd
import sys
sys.path.insert(0, "../../utils/")
from sklearn.model_selection import train_test_split
from training_models.regression_models import RegressionModels
from joblib import dump
import numpy as np

In [11]:
key_map = {
    "R2": "r2",
    "MAE": "neg_mean_absolute_error",
    "MSE": "neg_mean_squared_error"
}

In [12]:
def split(df_data, seed):
    #Separa los datos
    train_data, val_data = train_test_split(df_data, test_size=0.2, random_state=seed)
    return train_data, val_data

In [13]:
def train(train_v, validation_v, iteration, repr_name, seed):
    #Separa datos de sus target de entrenamiento y validacion
    train_values = train_v.drop(columns="target").values
    train_response = train_v["target"].values

    validation_values = validation_v.drop(columns="target").values
    validation_response = validation_v["target"].values

    print(f"Training model Random Forest, iteration: {iteration}")
    #Se instancia el objeto
    clf_model = RegressionModels(X_train=train_values, X_val=validation_values, y_train=train_response, y_val=validation_response)
    #Se entrena el respectivo algoritmo con k-fold
    clf_model.instance_random_forest()
    clf_model.process_model(kfold=True, k=5)
    print (clf_model.performances)

    #Se guarda el modelo
    dump(clf_model.model, f"../../models/RandomForest_regression_{iteration}_{repr_name}_seed{seed}.joblib")

    return clf_model.performances

In [14]:
def metrics(perf):
    #Se obtienen las metricas de entrenamiento y validacion en variables diferentes
    row = {}
    train_metrics = perf["training_metrics"]
    val_metrics = perf["validation_metrics"]
    #Renombra metricas
    train_renamed = {key_map.get(k, k): v for k, v in train_metrics.items()}
    for metric_name in key_map.values():
        row[f"Train_{metric_name}"] = round(train_renamed[metric_name], 4)
        row[f"Val_{metric_name}"] = round(val_metrics[metric_name], 4)
    return row

In [15]:
def main_train(df_data, repr_name, seed):
    all_metrics = []
    df_train, df_val= split(df_data, seed)
    perf_base = train(df_train, df_val, repr_name, seed)
    all_metrics.append(metrics(perf_base))

    df_metrics = pd.DataFrame(all_metrics)
    df_metrics.to_csv(f"../../models/metrics_{repr_name}_reg_RandomForest.csv", index=False)

In [16]:
repr_name="antiviral_homology_90_Q_prot5_embedding"
df_data = pd.read_csv(f"../../data/numerical_rep_reg/{repr_name}.csv")
df_data.drop(["experimental_characteristics"], axis=1, inplace=True)

In [17]:
folder = "../../data/numerical_rep/"
unique_seeds= [42]
#unique_seeds = np.random.choice(range(100), size=30, replace=False)
#unique_seeds = [94, 42, 98, 43, 90, 44, 99, 93, 66, 34, 72, 60, 6, 39, 26, 74, 17,8, 51, 96, 53, 13, 20, 33, 29, 65, 46, 82, 79, 89]

In [18]:
print(f"Processing {repr_name}")
metrics_path = f"../../models/metrics_regression_{repr_name}.csv"
seeds_used = unique_seeds
main_train(df_data, repr_name, '42')
print(f"Finished processing {repr_name}")
print("=====================================")

Processing antiviral_homology_90_Q_prot5_embedding


InvalidParameterError: The 'random_state' parameter of train_test_split must be an int in the range [0, 4294967295], an instance of 'numpy.random.mtrand.RandomState' or None. Got '42' instead.